# B2 direct-critic readiness probes (§1893)
Executes the frozen probe plan `docs/plans/2026-10-05-b2-critic-probes-plan.md` (peer-reviewed): P0 adjacent-rung floor, P1 convergence ledger + 2× continuation, P2 0.5×/2× width probes. Four foreground runs, sequential. Do not edit thresholds — they are frozen in the plan.

Preconditions: L4 GPU runtime, Drive mounted, this repo checked out on branch `opencode/b2-critic-probes`.

In [1]:
# Bootstrap (safe to re-run): mount Drive, fetch the probe branch, enter the repo.
import os, subprocess
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
script = '/content/probe_b2/scripts/run_allocator_optimization_benchmark.py'
if not os.path.isfile(script):
    subprocess.run(['rm', '-rf', '/content/probe_b2'], check=True)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch',
                    'opencode/b2-critic-probes',
                    'https://github.com/ROMUSKING/para_001.git',
                    '/content/probe_b2'], check=True)
%cd /content/probe_b2
!git log --oneline -1

Mounted at /content/drive
/content/probe_b2
55abbdf (grafted, HEAD -> opencode/b2-critic-probes, origin/opencode/b2-critic-probes) Probe notebook: automatic Drive run record (report, inputs, inventory)


In [2]:
import subprocess, sys
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout[-2000:]); print(r.stderr[-1000:], file=sys.stderr)
    return r
r = sh('nvidia-smi --query-gpu=name --format=csv,noheader')
assert 'L4' in r.stdout, 'L4 required, got: ' + r.stdout
import torch; assert torch.cuda.is_available(), 'torch.cuda unavailable'
print('torch', torch.__version__, '| GPU ok')

NVIDIA L4



torch 2.11.0+cu130 | GPU ok


In [3]:
import hashlib
DRIVE = '/content/drive/MyDrive/Colab Notebooks/AdjointRWM_Production'
RUN_ID = 'droid100_adjoint_v2_5seeds_20261001T080821Z'
OUT = '/content/drive/MyDrive/Colab Notebooks/AdjointRWM_Production/runs/b2_critic_probes_20261005'
import os
cache = os.path.join(DRIVE, 'runs', RUN_ID, 'cache')
assert os.path.isdir(cache), 'B2 cache missing on Drive: ' + cache
cfg = os.path.join(DRIVE, 'runs', RUN_ID, 'config', 'run_config.json')
assert os.path.isfile(cfg), 'run config missing: ' + cfg
ckpt0 = os.path.join(DRIVE, 'runs', RUN_ID, 'jobs', 'seed_0', 'dynamics', 'best.pt')
assert os.path.isfile(ckpt0), 'teacher checkpoint missing: ' + ckpt0
# Code identity: probe flags must exist in the checked-out script.
src = open('scripts/run_allocator_optimization_benchmark.py').read()
for flag in ('--critic-width-scale', '--critic-only', '--probe-panels', 'adjacent_rung_panels', 'surrogate_ledger'):
    assert flag in src, 'missing probe code: ' + flag
head = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('HEAD', head, '| data + code verified')
!mkdir -p "$OUT"/probe_w10 "$OUT"/probe_w05 "$OUT"/probe_w20 "$OUT"/probe_cont

HEAD 55abbdf | data + code verified


In [4]:
# Provenance (recorded into every output dir by the run script via OUT; this cell
# additionally freezes the code identity for the record). Fails loudly on mismatch.
import json as _json
head = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True,
                      cwd='/content/probe_b2').stdout.strip()
branch = subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True,
                        text=True, cwd='/content/probe_b2').stdout.strip()
prov = {'branch': branch, 'head': head,
        'torch': torch.__version__, 'cuda': torch.cuda.get_device_name(0),
        'run_id': RUN_ID, 'plan': 'docs/plans/2026-10-05-b2-critic-probes-plan.md'}
open(os.path.join(OUT, 'code_provenance.json'), 'w').write(_json.dumps(prov, indent=1))
print(_json.dumps(prov, indent=1))

{
 "branch": "opencode/b2-critic-probes",
 "head": "55abbdf4cb55eabc631c3e509e915dec5bba5b92",
 "torch": "2.11.0+cu130",
 "cuda": "NVIDIA L4",
 "run_id": "droid100_adjoint_v2_5seeds_20261001T080821Z",
 "plan": "docs/plans/2026-10-05-b2-critic-probes-plan.md"
}


## Run `w10`: width 1.0×, 2500 steps

In [ ]:
!python -u scripts/run_allocator_optimization_benchmark.py --critic-only --probe-panels --critic-width-scale 1.0 --eval-checkpoints 300 1000 2500 --max-steps 2500 --output-dir "$OUT"/probe_w10 --local-cache /content/cache 2>&1 | tee "$OUT"/probe_w10.log

Running Allocator Optimization Benchmark on device: cuda
Loading DROID-100 dataset...
Dataset loaded: 5910 train, 796 val, 835 test windows

  OPTIMIZING & BENCHMARKING SEED 0 / 5
/content/probe_b2/src/adjointrwm/models/adjoint_rwm.py:87: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal_encoder = nn.TransformerEncoder(encoder_layer, num_layers=config.transformer_layers)


## Run `w05`: width 0.5×, 2500 steps

In [ ]:
!python -u scripts/run_allocator_optimization_benchmark.py --critic-only --probe-panels --critic-width-scale 0.5 --eval-checkpoints 300 1000 2500 --max-steps 2500 --output-dir "$OUT"/probe_w05 --local-cache /content/cache 2>&1 | tee "$OUT"/probe_w05.log

## Run `w20`: width 2.0×, 2500 steps

In [ ]:
!python -u scripts/run_allocator_optimization_benchmark.py --critic-only --probe-panels --critic-width-scale 2.0 --eval-checkpoints 300 1000 2500 --max-steps 2500 --output-dir "$OUT"/probe_w20 --local-cache /content/cache 2>&1 | tee "$OUT"/probe_w20.log

## Run `cont`: width 1.0×, 5000 steps

In [ ]:
!python -u scripts/run_allocator_optimization_benchmark.py --critic-only --probe-panels --critic-width-scale 1.0 --eval-checkpoints 300 1000 2500 5000 --max-steps 5000 --output-dir "$OUT"/probe_cont --local-cache /content/cache 2>&1 | tee "$OUT"/probe_cont.log

## Verification (all four summaries)

In [ ]:
import hashlib, json, os
for tag in ('w10', 'w05', 'w20', 'cont'):
    p = os.path.join(OUT, f'probe_{tag}', 'b2_2_allocator_optimization_summary.json')
    raw = open(p, 'rb').read()
    s = json.loads(raw)
    assert s['seed_finite'] == [True]*5, (tag, s['seed_finite'])
    assert 'surrogate_ledger' in s and 'val_checkpoints' in s, tag
    last = s['eval_checkpoints'][-1]
    v = s['val_checkpoints'][str(last)]['mean_regrets']
    print(f"{tag}: sha256={hashlib.sha256(raw).hexdigest()[:16]} "
          f"width={s['critic_width_scale']} steps={last} "
          f"val critic={v['allocator_critic']:.5f} random={v['random_expected']:.5f} "
          f"unc={v['uncertainty']:.5f} mode0={v['always_mode0']:.5f}")
print('All four summaries present, finite, and structurally complete.')

## Save the executed record
Colab menu: **File → Save a copy in Drive**, into `MyDrive/Colab Notebooks/AdjointRWM_Production/runs/b2_critic_probes_20261005/`. The executed copy (with cell outputs) is part of the evidence trail alongside `code_provenance.json`, the per-seed partials, and the four summaries.

In [ ]:
# Automatic executed record (no manual save needed): consolidates provenance,
# verification, file inventory with hashes, and the full executed-input history
# onto Drive. The per-run stdout already streams to the tee'd logs.
import glob as _glob, hashlib as _hl, json as _js, os as _os, time as _time
inv = {}
for f in sorted(_glob.glob(_os.path.join(OUT, '**', '*'), recursive=True)):
    if _os.path.isfile(f):
        h = _hl.sha256()
        with open(f, 'rb') as fh:
            for b in iter(lambda: fh.read(1 << 20), b''):
                h.update(b)
        inv[_os.path.relpath(f, OUT)] = {'bytes': _os.path.getsize(f),
                                         'sha256': h.hexdigest()}
report = {'closed_utc': _time.strftime('%Y-%m-%dT%H:%M:%SZ', _time.gmtime()),
          'provenance': prov, 'file_inventory': inv}
open(_os.path.join(OUT, 'run_report.json'), 'w').write(_js.dumps(report, indent=1))
hm = get_ipython().history_manager
with open(_os.path.join(OUT, 'session_inputs.py'), 'w') as fh:
    fh.write('# Executed inputs, in order (outputs: see tee\'d probe_*.log files).\n')
    for _s, _n, entry in hm.get_range(hm.session_number):
        fh.write('# ' + '-' * 60 + '\n' + entry.rstrip() + '\n')
print('run_report.json + session_inputs.py written;', len(inv), 'files inventoried')

Done. `run_report.json` (hashes of every artefact), `session_inputs.py` (every executed input), the four summaries/partials, and the tee'd logs are all on Drive under the run folder — nothing further needs saving, including the notebook itself.